# 02 — Feature Engineering Demo

Minh họa cách tạo 5 nhóm feature từ dữ liệu thô.

**Mục tiêu:**
1. Hiểu cách tạo từng nhóm feature
2. Xem feature nào quan trọng (correlation với target)
3. Thực hành feature selection

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Tải dữ liệu gốc

In [ ]:
from src.data.loader import load_raw_data, create_temporal_split, prepare_targets

df = load_raw_data()
print(f'Du lieu goc: {df.shape[1]} cot, {df.shape[0]} dong')
print(f'Cac cot: {list(df.columns)}')

## 2. Tạo feature theo từng nhóm

### Nhóm 1: Lag Features

In [ ]:
from src.features.engineering import create_lag_features

df_lag = create_lag_features(df, columns=['water_level', 'sea_level'], lag_periods=[1, 7, 14])
print(f'Sau khi them lag features: {df_lag.shape[1]} cot')
df_lag[['water_level', 'water_level_lag_1', 'water_level_lag_7', 'water_level_lag_14']].head(20)

### Nhóm 2: Rolling Features

In [ ]:
from src.features.engineering import create_rolling_features

df_roll = create_rolling_features(df, windows=[3, 7])
print(f'Sau khi them rolling features: {df_roll.shape[1]} cot')
df_roll[['water_level', 'rolling_mean_7', 'rolling_std_7']].tail(10)

### Nhóm 3: Seasonal Features

In [ ]:
from src.features.engineering import create_seasonal_features

df_season = create_seasonal_features(df)
print(f'Sau khi them seasonal features: {df_season.shape[1]} cot')

# Ve chu ky trieu
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df_season.index[:100], df_season['tidal_cycle_sin'][:100], label='tidal_cycle_sin')
ax.plot(df_season.index[:100], df_season['tidal_cycle_cos'][:100], label='tidal_cycle_cos')
ax.set_ylabel('Gia tri')
ax.set_title('Ma hoa chu ky trieu (~14.76 ngay)')
ax.legend()
plt.tight_layout()
plt.show()

### Nhóm 4: Interaction Features

In [ ]:
from src.features.engineering import create_interaction_features

df_inter = create_interaction_features(df)
print(f'Sau khi them interaction features: {df_inter.shape[1]} cot')
df_inter[['water_level', 'sea_river_diff', 'flood_pressure']].describe()

### Nhóm 5: Risk Features

In [ ]:
from src.features.engineering import create_risk_features

df_risk = create_risk_features(df)
print(f'Sau khi them risk features: {df_risk.shape[1]} cot')
df_risk[['water_level', 'distance_to_threshold', 'delta_1', 'acceleration', 'is_rising']].tail(10)

## 3. Build toàn bộ feature

In [ ]:
from src.features.engineering import build_all_features

# Tao target cho cac horizon
df_with_targets = prepare_targets(df, horizons=[1, 3, 7])

# Tao tat ca feature
df_features = build_all_features(df_with_targets)
print(f'Tong so feature: {df_features.shape[1]} cot')
print(f'Sau khi drop NaN: {df_features.dropna().shape[0]} dong con lai')

## 4. Feature Selection

In [ ]:
from src.data.loader import create_temporal_split
from src.features.engineering import select_features_by_correlation

# Chia du lieu
splits = create_temporal_split(df_features.dropna())
train = splits['train']

# Xac dinh cot feature (khong phai target hay cot goc)
target_cols = [c for c in train.columns if c.startswith('water_level_t')]
raw_cols = ['river_flow_a', 'river_flow_b', 'sea_level', 'water_level']
feature_cols = [c for c in train.columns if c not in target_cols + raw_cols + ['month', 'day_of_year']]

# Chon feature cho horizon t+1
X_train = train[feature_cols]
y_train = train['water_level_t1']

selected = select_features_by_correlation(X_train, y_train)
print(f'Tong feature: {len(feature_cols)}')
print(f'Sau khi chon: {len(selected)}')
print(f'Cac feature bi loai: {[f for f in feature_cols if f not in selected][:10]}...')

In [ ]:
# Tinh tuong quan cua tung feature voi target
target_corr = X_train[selected].corrwith(y_train, method='spearman').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
target_corr.head(20).plot(kind='barh', ax=ax)
ax.set_xlabel('|Spearman Correlation| voi target t+1')
ax.set_title('Top 20 feature tuong quan cao nhat voi target')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Tóm tắt

**Bài học:**
1. Feature engineering tạo ra nhiều feature có ý nghĩa thủy văn
2. Feature selection giúp loại bỏ feature dư thừa và yếu
3. Lag features (đặc biệt lag 14 ngày) có tương quan cao với target t+1
4. Feature engineering là bước quan trọng nhất trong pipeline!